# Image CLI Optimizer

This notebook accompanies Chapter 11 §11.2.1–11.2.3 of *Context Engineering with DSPy*: building and optimizing the image-population CLI that fills the placeholder image slots emitted by the landing-page skill from §11.1.

The CLI has three optimizable DSPy predictors — extract image briefs, write image-generation prompts, and score images against the brand — plus an OpenAI image-generation API call. The predictors are optimized with `dspy.GEPA`; image generation remains a deterministic module step driven by the optimized prompt writer.

## Required environment variables

Add this to a `.env` at the repo root (see `.env.example`):

- `OPENAI_API_KEY` — used for the text predictors, reflection LM, visual judge, and image-generation API.

No coding-agent CLI is required to run this notebook.

## Required assets

The seed-failure dataset references `acme_landing.html` and `outpost_landing.html` — pages produced by the landing-page skill from §11.1. Drop your own under `./assets/landing-pages/`. We don't ship sample HTML in the repo because the dataset only does its job when it's *your* real failed pages.


In [ ]:
%pip install -r ../requirements.txt -q

## Setup

Load env vars and configure DSPy. We default to `openai/gpt-5.6-luna` for the text predictors because the point of §11.2.3 is to optimize the prompts until a cheaper runtime model approaches strong-model quality. Image generation is called through the OpenAI SDK because `dspy.LM` returns language-model completions, not a `dspy.Image`.


In [ ]:
import base64
import os

import dspy
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY not in env"

dspy.configure(lm=dspy.LM("openai/gpt-5.6-luna"))


## Step 1: three signatures, one module

Each DSPy predictor gets its own `dspy.Signature`, which gives GEPA a separate instruction slot and `pred_name` for reflection-driven mutation. The OpenAI image-generation call is a deterministic step inside the module; the `WriteImagePrompt` predictor is the component GEPA can improve to change its output.


In [ ]:
class ExtractImageBriefs(dspy.Signature):
    """Pull the image slots out of the page HTML.

    For each <img> with a data-image-brief attribute, return the brief,
    the surrounding section's purpose (hero, feature, testimonial), and
    the aspect ratio the layout expects.
    """
    page_html: str = dspy.InputField()
    slots: list[dict] = dspy.OutputField(
        desc="List of {slot_id, brief, section_role, aspect_ratio}"
    )


class WriteImagePrompt(dspy.Signature):
    """Turn a single image brief + the brand guide into a generation
    prompt for the image model. Emphasize the brand palette, mood, and
    style. Avoid asking for text inside the image."""
    brief:       str  = dspy.InputField()
    brand_guide: dict = dspy.InputField()
    section_role: str = dspy.InputField()
    image_prompt: str = dspy.OutputField()


class ScoreImageAgainstBrand(dspy.Signature):
    """Judge whether the generated image matches the brand guide.
    Return a score and concrete feedback the next attempt can act on."""
    brand_guide:   dict        = dspy.InputField()
    brief:         str         = dspy.InputField()
    image:         dspy.Image  = dspy.InputField()
    score:         float       = dspy.OutputField(desc="0.0 to 1.0")
    feedback:      str         = dspy.OutputField(
        desc="What's off about this image and how to fix it"
    )


## Step 2: the `ImagePopulator` module

`dspy.Refine` requires a `dspy.Module` and a two-argument reward function. `SlotGenerator` supplies that module boundary. Refine runs it up to three times, keeps the best result by score, and feeds prior feedback into the next attempt. The final module also retains per-slot scores and feedback so the GEPA metric has real quality signal instead of merely checking whether any image was returned.


In [ ]:
def _openai_image_size(aspect_ratio: str) -> str:
    """Map layout aspect-ratio labels to sizes accepted by the Images API."""
    ratio = str(aspect_ratio).strip().lower()
    if ratio in {"16:9", "3:2", "landscape", "wide"}:
        return "1536x1024"
    if ratio in {"9:16", "2:3", "portrait", "tall"}:
        return "1024x1536"
    return "1024x1024"


def generate_image(prompt: str, model: str, aspect_ratio: str) -> dspy.Image:
    """Generate an image through OpenAI's image endpoint and wrap it for DSPy."""
    response = OpenAI().images.generate(
        model=model,
        prompt=prompt,
        size=_openai_image_size(aspect_ratio),
        quality="medium",
    )
    image_data = response.data[0]
    if image_data.b64_json:
        return dspy.Image(base64.b64decode(image_data.b64_json))
    if image_data.url:
        return dspy.Image(image_data.url)
    raise ValueError("Image API returned neither b64_json nor url")


def slot_reward(args, pred):
    return pred.score


class SlotGenerator(dspy.Module):
    """Generate and self-score one image for a single placeholder slot."""
    def __init__(self, image_model="openai/gpt-image-2"):
        super().__init__()
        self.write_prompt = dspy.ChainOfThought(WriteImagePrompt)
        self.score = dspy.Predict(ScoreImageAgainstBrand)
        self.image_model = image_model

    def forward(self, slot, brand_guide):
        prompt = self.write_prompt(
            brief=slot["brief"], brand_guide=brand_guide,
            section_role=slot["section_role"],
        ).image_prompt
        image = generate_image(prompt, self.image_model, slot["aspect_ratio"])
        judgment = self.score(
            brand_guide=brand_guide,
            brief=slot["brief"],
            image=image,
        )
        return dspy.Prediction(
            image=image,
            score=judgment.score,
            feedback=judgment.feedback,
        )


class ImagePopulator(dspy.Module):
    """Fill the placeholder image slots in a landing page."""
    def __init__(self, image_model="openai/gpt-image-2"):
        super().__init__()
        self.extract = dspy.ChainOfThought(ExtractImageBriefs)
        self.generate = dspy.Refine(
            module=SlotGenerator(image_model),
            N=3,
            reward_fn=slot_reward,
            threshold=0.8,
        )

    def forward(self, page_html, brand_guide):
        slots = self.extract(page_html=page_html).slots
        filled, slot_scores, slot_feedback = {}, {}, {}
        for slot in slots:
            result = self.generate(slot=slot, brand_guide=brand_guide)
            slot_id = slot["slot_id"]
            filled[slot_id] = result.image
            slot_scores[slot_id] = result.score
            slot_feedback[slot_id] = result.feedback
        return dspy.Prediction(
            filled_slots=filled,
            slot_scores=slot_scores,
            slot_feedback=slot_feedback,
        )


## Step 3: CLI seed failures

These are *image-CLI* failures, not page-layout failures — the SKILL.md was fine, the images it slotted weren't. Note them as you hit them: hero filled with a generic stock-photo, color cast that doesn't match the brand, an image with embedded text the model couldn't spell. The references to `acme_landing.html` etc. should point at your own pages.


In [ ]:
cli_seed_failures = [
    {
        "page_html":   open("./assets/landing-pages/acme_landing.html").read(),
        "brand_guide": {"colors": {"primary": "#4338CA"}, "mood": "minimal"},
        "failure_mode": "stock_photo_vibes",
        "notes": "Hero filled with a generic shaking-hands photo. Brand "
                 "guide said minimal/abstract.",
    },
    {
        "page_html":   open("./assets/landing-pages/outpost_landing.html").read(),
        "brand_guide": {"colors": {"primary": "#2D5016"}, "mood": "warm, human"},
        "failure_mode": "color_cast_off",
        "notes": "Feature images had a blue cast; brand is forest green.",
    },
    # ... 3-8 more like this.
]


## Step 4: expand the dataset synthetically

CLI-layer rollouts are cheap enough to grow the task set by an order of magnitude. `SyntheticImageTaskGenerator` creates more `(page_html, brand_guide, failure_mode)` tasks targeting the same image-fit failures. The metric below scores generated images directly and therefore does not pretend that an LM can synthesize trustworthy gold image files.


In [ ]:
class SyntheticImageTaskGenerator(dspy.Signature):
    """Given real failed runs of the image-population CLI, generate
    additional page + brand-guide pairs that target the same kinds of
    image-fit failures.
    """
    seed_failures: list[dict] = dspy.InputField(
        desc="Real failed runs: page_html, brand_guide, what went wrong"
    )
    num_to_generate: int = dspy.InputField()

    new_tasks: list[dict] = dspy.OutputField(
        desc="List of {page_html, brand_guide, failure_mode, notes} dicts"
    )


generate = dspy.ChainOfThought(SyntheticImageTaskGenerator)
synthetic = generate(
    seed_failures=cli_seed_failures,
    num_to_generate=50,
).new_tasks

examples = [
    dspy.Example(**task).with_inputs("page_html", "brand_guide")
    for task in cli_seed_failures + synthetic
]

# With the recommended ten real failures plus fifty synthetic tasks, this
# leaves 50 train examples and 10 held-out validation examples.
valset = examples[-10:]
trainset = examples[:-10]


## Step 5: optimize the prompts until the cheap model is good enough to deploy

The image CLI runs on every landing-page generation, forever. Per-call cost compounds, so the experiment uses the chapter's cheaper runtime choices — `openai/gpt-5.6-luna` for text predictors and `openai/gpt-image-1.5` for image generation — while keeping `openai/gpt-5.6-sol` as the strong reflection LM.

The image model is passed to `ImagePopulator` explicitly. `dspy.context` controls DSPy predictors but has no `image_lm` setting consumed by this module.

This cell uses `max_metric_calls=3` to smoke-test wiring. The next cell is the full-budget version (commented out) with `max_metric_calls=600`.


In [ ]:
def cli_metric(example, prediction, trace=None, pred_name=None, pred_trace=None):
    """Average the module's per-slot visual-judge scores and return feedback."""
    slot_scores = getattr(prediction, "slot_scores", None) or {}
    slot_feedback = getattr(prediction, "slot_feedback", None) or {}
    if not slot_scores:
        return dspy.Prediction(
            score=0.0,
            feedback="No image slots were extracted or filled.",
        )

    score = sum(float(value) for value in slot_scores.values()) / len(slot_scores)
    details = "\n".join(
        f"{slot_id}: score={float(slot_scores[slot_id]):.3f}; "
        f"feedback={slot_feedback.get(slot_id, 'none')}"
        for slot_id in sorted(slot_scores)
    )
    failure_mode = example.get("failure_mode", "unspecified")
    return dspy.Prediction(
        score=score,
        feedback=f"Target failure mode: {failure_mode}\n{details}",
    )


with dspy.context(lm=dspy.LM("openai/gpt-5.6-luna")):
    optimizer = dspy.GEPA(
        metric=cli_metric,
        reflection_lm=dspy.LM(
            "openai/gpt-5.6-sol",
            temperature=1.0,
            max_tokens=8000,
        ),
        max_metric_calls=3,
        reflection_minibatch_size=2,
        candidate_selection_strategy="pareto",
        num_threads=2,
        track_stats=True,
        log_dir="./gepa_logs",
    )
    optimized = optimizer.compile(
        student=ImagePopulator(image_model="openai/gpt-image-1.5"),
        trainset=trainset,
        valset=valset,
    )

print("done — best score:", getattr(optimized, "score", None))


In [ ]:
# Full reproduction of the book's §11.2.3 run.
# Uncomment to run — ~600 rollouts at cents each.
#
# with dspy.context(lm=dspy.LM("openai/gpt-5.6-luna")):
#     optimizer = dspy.GEPA(
#         metric=cli_metric,
#         reflection_lm=dspy.LM(
#             "openai/gpt-5.6-sol",
#             temperature=1.0,
#             max_tokens=8000,
#         ),
#         max_metric_calls=600,
#         reflection_minibatch_size=8,
#         candidate_selection_strategy="pareto",
#         num_threads=6,
#         track_stats=True,
#         log_dir="./gepa_logs",
#     )
#     optimized = optimizer.compile(
#         student=ImagePopulator(image_model="openai/gpt-image-1.5"),
#         trainset=trainset,
#         valset=valset,
#     )
#
# optimized.save("image_populator_optimized.json")
#
# # Deploy on the same models GEPA optimized against.
# production = ImagePopulator(image_model="openai/gpt-image-1.5")
# production.load_state(optimized.dump_state())


The optimized CLI inherits the landing-page skill's improvements for free: the skill emits placeholders, the CLI fills them, and the page that ships to the user has been improved at two layers in the same optimization pass.
